In [4]:
import numpy as np
from PIL import Image, ImageFilter
from tqdm import tqdm

In [5]:
SUPPORTED_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

CLASS_TO_LABEL = {
    "no_fall": 0,
    "fall": 1
}

def get_image_paths(split_directory):
    samples = []

    for class_name, label in CLASS_TO_LABEL.items():
        class_directory = split_directory / class_name

        for image_path in class_directory.rglob("*"):
            if (
                image_path.is_file()
                and image_path.suffix.lower() in SUPPORTED_EXTENSIONS
            ):
                samples.append((image_path, label))

    return samples

train_samples = get_image_paths(TRAIN_DIR)
val_samples = get_image_paths(VAL_DIR)

print("Training samples:", len(train_samples))
print("Validation samples:", len(val_samples))

Training samples: 1123
Validation samples: 361


In [6]:
def create_blurred_feature(
    image_path,
    resize_size=(64, 64),
    final_size=(16, 16),
    blur_radius=12
):
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        image = image.resize(resize_size)
        image = image.filter(
            ImageFilter.GaussianBlur(radius=blur_radius)
        )
        image = image.resize(final_size)

        image_array = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        return image_array.flatten()

In [7]:
def build_feature_dataset(samples):
    features = []
    labels = []

    for image_path, label in tqdm(samples):
        feature = create_blurred_feature(image_path)

        features.append(feature)
        labels.append(label)

    return (
        np.asarray(features),
        np.asarray(labels)
    )

X_train_blur, y_train = build_feature_dataset(train_samples)

X_val_blur, y_val = build_feature_dataset(val_samples)

print(X_train_blur.shape)
print(X_val_blur.shape)

100%|██████████| 361/361 [00:00<00:00, 428.47it/s]

(1123, 768)
(361, 768)


In [8]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

model.fit(X_train_blur, y_train)

print("Training Complete")

Training Complete


In [10]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score
)

preds = model.predict(X_val_blur)

print(
    "Ordinary Accuracy:",
    accuracy_score(y_val, preds)
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(y_val, preds)
)

Ordinary Accuracy: 0.554016620498615
Balanced Accuracy: 0.4903950302644154
